# X Engagement & Activity Tracker
This notebook tracks engagement and activity for your X (formerly Twitter) account using the X API.

In [1]:
import os
import tweepy
import pandas as pd
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

X_API_KEY = os.getenv("X_API_KEY")

if X_API_KEY:
    print("X_API_KEY loaded successfully.")
else:
    print("X_API_KEY not found. Please check your .env file.")

X_API_KEY loaded successfully.


In [2]:
# Initialize the X (Twitter) Client
client = tweepy.Client(bearer_token=X_API_KEY)

try:
    # Get information about the authenticated user
    # Note: get_me() requires OAuth 1.0a or OAuth 2.0 User Context. 
    # With just a Bearer Token, we might need to search by username or use other endpoints.
    # Let's try to get a sample user or the user associated with the token if possible.
    
    # Since Bearer Token is usually for App-only access, let's try to get a specific user (you can change this to your username)
    # If you know your username, replace 'TwitterDev' with it.
    username = 'TwitterDev' # Placeholder
    user = client.get_user(username=username, user_fields=['public_metrics', 'description', 'created_at'])
    
    if user.data:
        print(f"Successfully connected! Info for @{user.data.username}:")
        print(f"Name: {user.data.name}")
        print(f"Followers: {user.data.public_metrics['followers_count']}")
        print(f"Following: {user.data.public_metrics['following_count']}")
        print(f"Tweets: {user.data.public_metrics['tweet_count']}")
    else:
        print("Could not find user data. Please check if the username is correct.")
except Exception as e:
    print(f"Error connecting to X API: {e}")

Error connecting to X API: 402 Payment Required
Your enrolled account [2044171188864204800] does not have any credits to fulfill this request.


# User Engagement & Activity Tracking
To track your engagement, we'll fetch your profile metrics and recent tweets. 
Note: Replace 'YOUR_USERNAME' with your actual X handle.

In [ ]:
USERNAME = 'YOUR_USERNAME' # <-- Update this with your X username

def get_user_stats(username):
    try:
        user = client.get_user(username=username, user_fields=['public_metrics', 'description'])
        if user.data:
            metrics = user.data.public_metrics
            stats = {
                'Followers': metrics['followers_count'],
                'Following': metrics['following_count'],
                'Tweets': metrics['tweet_count'],
                'Listed': metrics['listed_count']
            }
            return stats
        else:
            return None
    except Exception as e:
        print(f"Error fetching user stats: {e}")
        return None

stats = get_user_stats(USERNAME)
if stats:
    df_stats = pd.DataFrame([stats])
    print("Current Account Stats:")
    display(df_stats)

In [ ]:
# Fetch Recent Tweets and Engagement
def get_recent_tweet_engagement(username, max_results=10):
    try:
        # Get user ID first
        user = client.get_user(username=username)
        if not user.data:
            return None
        
        user_id = user.data.id
        
        # Get tweets
        tweets = client.get_users_tweets(
            id=user_id, 
            max_results=max_results,
            tweet_fields=['public_metrics', 'created_at', 'text']
        )
        
        if not tweets.data:
            return None
        
        tweet_data = []
        for tweet in tweets.data:
            metrics = tweet.public_metrics
            tweet_data.append({
                'Date': tweet.created_at,
                'Tweet': tweet.text[:50] + '...',
                'Retweets': metrics['retweet_count'],
                'Replies': metrics['reply_count'],
                'Likes': metrics['like_count'],
                'Quotes': metrics['quote_count'],
                'Impressions': metrics.get('impression_count', 0)
            })
            
        return pd.DataFrame(tweet_data)
    except Exception as e:
        print(f"Error fetching tweets: {e}")
        return None

df_engagement = get_recent_tweet_engagement(USERNAME)
if df_engagement is not None:
    print("Recent Tweet Engagement:")
    display(df_engagement)

In [ ]:
# Visualization of Engagement
import plotly.express as px

if df_engagement is not None and not df_engagement.empty:
    # Melt the dataframe for easier plotting with Plotly
    df_melted = df_engagement.melt(id_vars=['Date', 'Tweet'], 
                                   value_vars=['Likes', 'Retweets', 'Replies', 'Quotes'],
                                   var_name='Metric', value_name='Count')
    
    fig = px.bar(df_melted, x='Date', y='Count', color='Metric', barmode='group',
                 title='Recent Tweet Engagement Metrics',
                 labels={'Count': 'Engagement Count', 'Date': 'Post Date'})
    fig.show()
else:
    print("No engagement data available to visualize. (API access required)")

In [ ]:
# Engagement Summary
if df_engagement is not None and not df_engagement.empty:
    total_likes = df_engagement['Likes'].sum()
    total_retweets = df_engagement['Retweets'].sum()
    avg_likes = df_engagement['Likes'].mean()
    
    print(f"Summary for the last {len(df_engagement)} tweets:")
    print(f"- Total Likes: {total_likes}")
    print(f"- Total Retweets: {total_retweets}")
    print(f"- Average Likes per Tweet: {avg_likes:.2f}")
else:
    print("Summary not available due to missing data.")